## Автоматическое определение необходимости эквализации изображения
Данные скрипты используют свойства гистограммы изображения для автоматического (без участия человека) принятия решения о необходимости эквализации изображения.

Использованные изображения: [BSDS300](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/bsds/)

Пусть критерием "выровненного" изображения будет являться (приблизительная) линейность эмпирической функции распределения (т. е. интеграла от гистограммы).
Это означает, что распределение цветов в самой гистограмме более-менее равномерно — как первая производная линейной функции, это будет функция с постоянным значением. Тогда производной от этой функции (вторая производная от эмпирической CDF) будет ноль (опять же, приблизительно).
Соответственно, два варианта определения "выровненности" изображения:
1) По производной: производная гистограммы в основном имеет очень малое значение, без резких скачков, особенно в начале и в конце диапазона яркостей
2) По самой гистограмме: примерно одинаковое количество пикселей на каждом отрезке диапазона яркостей.

### Определение по производной

Изображение можно считать "корректным", если, например, производная на протяжении всего диапазона будет иметь сравнительно низкие (по модулю) значения.

Насколько низкие? Пока не очень понятно.

Для идеального случая гистограммы CDF будет иметь прирост в N / 256 на каждый оттенок (N — число пикселей изображения), то есть каждое значение гистограммы равно N / 256. Допустим — для примера тогда (условно и несколько волюнтаристским образом) возьмём 5% от N / 256 как максимальное допустимое значение производной.

Возможно, также придётся "сглаживать" производную, используя, например, скользящее среднее: для случая, если есть большие пустые промежутки между значениями яркости (хотя обычно это встречается в уже эквализованных изображениях).

In [18]:
# папка с изображениями
FOLDER = './_stuff';
# индексы (с 0) изображений, которые будут показываться
IMAGES = [1234, 123412, 2,13, 12,4321 ,1225];
# пороговое отклонение (в долях от 1) значения производной от ???
DERIVATIVE_THRESHOLD = 0.05;
# размер окна для сглаживания (скользящее среднее)
WINDOW_SIZE = 5;

In [ ]:
import os;
import cv2 as cv;
import numpy as np;
import matplotlib.pyplot as plt;


def makeCdf(histogram):
    return np.cumsum(histogram);


# Получаем список файлов
files = sorted([
    os.path.join(FOLDER, name)
    for name in os.listdir(FOLDER)]);


for index in IMAGES:
    image_path = files[index % len(files)];

    img_bgr = cv.imread(image_path);

    gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY);

    # Гистограмма яркости
    histogram = cv.calcHist([gray], [0], None, [256], [0, 256]).flatten();

    # CDF
    cdf = makeCdf(histogram);

    # Первая производная гистограммы: разность соседних значений
    derivative = np.diff(histogram);

    bins = np.arange(256);
    derivative_bins = np.arange(1, 256);

    fig, ax = plt.subplots(2, 2, figsize=(14, 8));

    # Изображение
    ax[0, 0].imshow(gray, cmap='gray', vmin=0, vmax=255);
    ax[0, 0].set_title(os.path.basename(image_path));
    ax[0, 0].axis('off');

    # Гистограмма и CDF
    ax[0, 1].bar(bins, histogram, color='black', alpha=0.4);
    ax[0, 1].set_title('Гистограмма и CDF');
    ax[0, 1].set_ylabel('Количество пикселей');
    ax[0, 1].set_xlim(0, 255);

    ax_cdf = ax[0, 1].twinx();
    ax_cdf.plot(bins, cdf, color='red');
    ax_cdf.set_ylabel('CDF');

    # Гистограмма и производная
    ax[1, 1].bar(bins, histogram, color='black', alpha=0.4);
    ax[1, 1].set_title('Гистограмма и производная');
    ax[1, 1].set_xlabel('Уровень яркости');
    ax[1, 1].set_ylabel('Количество пикселей');
    ax[1, 1].set_xlim(0, 255);

    ax_derivative = ax[1, 1].twinx();
    ax_derivative.plot(derivative_bins, derivative, color='blue');
    ax_derivative.set_ylabel('Производная гистограммы');
    ax_derivative.axhline(0, color='red', linewidth=0.7, alpha=0.4);

    # Четвёртый график пока пустой
    ax[1, 0].axis('off');

    fig.tight_layout();
    plt.show();